# 01 · Pytest Core & Assertions

**Focus:** test discovery, naming conventions, and the power of the plain `assert` statement.

`pytest` is the de-facto standard for testing Python. Two things make it feel effortless
compared to the standard-library `unittest`:

1. **Zero boilerplate discovery.** Any file named `test_*.py` (or `*_test.py`), any function
   named `test_*`, is collected and run automatically. No classes, no `TestCase` inheritance.
2. **Assertion introspection.** You write ordinary `assert x == y`. When it fails, pytest
   *rewrites* the assertion under the hood to show you the actual values on both sides —
   something `unittest` only gives you if you remember the right `assertEqual` / `assertAlmostEqual`
   method.

> **The Jupyter pattern used throughout this course**
> `pytest` is a command-line tool, so we don't call it from Python directly. Instead every demo
> **(1)** writes the code to a real `.py` file with the `%%writefile` magic, then
> **(2)** runs it from a shell cell with `!pytest`. That is exactly how you'd run tests in a real project.

### Setup — point the shell at this course's tools
The `!` cells below run command-line tools (`pytest`, later `mutmut`, `playwright`). We prepend the
active kernel's `bin/` directory to `PATH` so those commands resolve to the versions installed for
**this course**, not some other Python on your machine. Run this cell first.

In [1]:
import sys, os
# The kernel's own interpreter lives in the course virtualenv's bin/ directory.
os.environ["PATH"] = os.path.dirname(sys.executable) + os.pathsep + os.environ["PATH"]
print("Using Python:", sys.executable)
!pytest --version

Using Python: /Users/ajinkyak/Desktop/JULY-WORK/a - training_assignments/explorable_codelesson/.venv/bin/python


pytest 9.1.1


### The code under test
A tiny calculator. Note `%%writefile` — it saves the cell to disk instead of executing it.

In [2]:
%%writefile nb01_calculator.py
def add(a, b):
    return a + b

def subtract(a, b):
    return a - b

def multiply(a, b):
    return a * b

def divide(a, b):
    if b == 0:
        raise ZeroDivisionError("cannot divide by zero")
    return a / b

Writing nb01_calculator.py


### The tests
Just functions named `test_*` that use the plain `assert` keyword.

In [3]:
%%writefile test_nb01.py
from nb01_calculator import add, subtract, multiply, divide

def test_add():
    assert add(2, 3) == 5

def test_subtract():
    assert subtract(10, 4) == 6

def test_multiply():
    assert multiply(3, 4) == 12

def test_divide():
    assert divide(10, 2) == 5

Writing test_nb01.py


### Run them
`-v` (verbose) lists every test and its PASS/FAIL status.

In [4]:
!pytest -v test_nb01.py

============================= test session starts ==============================
platform darwin -- Python 3.12.11, pytest-9.1.1, pluggy-1.6.0 -- /Users/ajinkyak/Desktop/JULY-WORK/a - training_assignments/explorable_codelesson/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/ajinkyak/Desktop/JULY-WORK/a - training_assignments/explorable_codelesson
plugins: syrupy-5.5.3, mock-3.15.1, cov-7.1.0, xdist-3.8.0, asyncio-1.4.0, time-machine-3.2.0, anyio-4.14.2, base-url-2.1.0, playwright-0.8.0
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collected 4 items                                                              

test_nb01.py::test_add PASSED                                            [ 25%]
test_nb01.py::test_subtract PASSED                                       [ 50%]
test_nb01.py::test_multiply PASSED                                       [ 75%]
test_nb01.py::test_divide PASSED                         

### Assertion introspection — the killer feature
Let's write a test that **fails on purpose** so you can see what pytest prints. Notice it shows you
the *left* value, the *right* value, and even the difference — all from a bare `assert`.

In [5]:
%%writefile test_nb01_fail.py
from nb01_calculator import add, multiply

def test_add_is_introspected():
    result = add(2, 3)
    assert result == 6          # WRONG on purpose: 5 != 6

def test_list_diff_is_introspected():
    assert [multiply(2, 2), multiply(3, 3)] == [4, 8]   # WRONG: 9 != 8

Writing test_nb01_fail.py


In [6]:
!pytest -v test_nb01_fail.py

============================= test session starts ==============================
platform darwin -- Python 3.12.11, pytest-9.1.1, pluggy-1.6.0 -- /Users/ajinkyak/Desktop/JULY-WORK/a - training_assignments/explorable_codelesson/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/ajinkyak/Desktop/JULY-WORK/a - training_assignments/explorable_codelesson
plugins: syrupy-5.5.3, mock-3.15.1, cov-7.1.0, xdist-3.8.0, asyncio-1.4.0, time-machine-3.2.0, anyio-4.14.2, base-url-2.1.0, playwright-0.8.0
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collecting ... 

collected 2 items                                                              

test_nb01_fail.py::test_add_is_introspected 

FAILED                       [ 50%]
test_nb01_fail.py::test_list_diff_is_introspected FAILED                 [100%]

=================================== FAILURES ===================================
___________________________ test_add_is_introspected ___________________________

    def test_add_is_introspected():
        result = add(2, 3)
>       assert result == 6          # WRONG on purpose: 5 != 6
        ^^^^^^^^^^^^^^^^^^
E       assert 5 == 6

test_nb01_fail.py:5: AssertionError
________________________ test_list_diff_is_introspected ________________________

    def test_list_diff_is_introspected():
>       assert [multiply(2, 2), multiply(3, 3)] == [4, 8]   # WRONG: 9 != 8
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
E       AssertionError: assert [4, 9] == [4, 8]
E         
E         At index 1 diff: 9 != 8
E         
E         Full diff:
E           [
E               4,
E         -     8,...
E         
E         ...Full output truncated (4 lines hidden), use '-

Read that output: `assert 5 == 6`, and for the list it points at the exact differing index.
The equivalent in `unittest` requires you to *choose* `assertEqual` to get a diff; a plain
`assert result == 6` inside a `unittest.TestCase` would just say `AssertionError` with **no values**.
That is why pytest's plain `assert` is such a productivity win.

### Selecting tests by name with `-k`
`-k` runs only tests whose name matches an expression. Great for iterating on one failure.

In [7]:
!pytest -v -k "test_add" test_nb01.py

============================= test session starts ==============================
platform darwin -- Python 3.12.11, pytest-9.1.1, pluggy-1.6.0 -- /Users/ajinkyak/Desktop/JULY-WORK/a - training_assignments/explorable_codelesson/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/ajinkyak/Desktop/JULY-WORK/a - training_assignments/explorable_codelesson
plugins: syrupy-5.5.3, mock-3.15.1, cov-7.1.0, xdist-3.8.0, asyncio-1.4.0, time-machine-3.2.0, anyio-4.14.2, base-url-2.1.0, playwright-0.8.0
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collected 4 items / 3 deselected / 1 selected                                  

test_nb01.py::test_add PASSED                                            [100%]

======================= 1 passed, 3 deselected in 0.01s ========================


### ⚠️ Common pitfall — the assert-tuple trap
Never write `assert (value == expected, "message")`. The parentheses make it a **tuple**, and a
non-empty tuple is *always truthy*, so the assertion **can never fail** — your test silently passes
no matter what. The correct form is `assert value == expected, "message"` (no parentheses).

### 🔬 Try it yourself
Below is the trap in action. **Predict first:** the math is wrong (`5 == 6`), so should this test
pass or fail? Run it and read pytest's warning carefully. Then fix it by removing the parentheses and
re-run — now it fails as it should.

In [8]:
%%writefile test_nb01_tryit.py
from nb01_calculator import add

def test_tuple_trap():
    # BUG: trailing comma via parentheses -> this is a (bool, str) TUPLE, always truthy.
    assert (add(2, 3) == 6, "add is broken")   # <-- try deleting the parentheses and re-running

Writing test_nb01_tryit.py


In [9]:
!pytest -v test_nb01_tryit.py

============================= test session starts ==============================
platform darwin -- Python 3.12.11, pytest-9.1.1, pluggy-1.6.0 -- /Users/ajinkyak/Desktop/JULY-WORK/a - training_assignments/explorable_codelesson/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/ajinkyak/Desktop/JULY-WORK/a - training_assignments/explorable_codelesson
plugins: syrupy-5.5.3, mock-3.15.1, cov-7.1.0, xdist-3.8.0, asyncio-1.4.0, time-machine-3.2.0, anyio-4.14.2, base-url-2.1.0, playwright-0.8.0
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collected 1 item                                                               

test_nb01_tryit.py::test_tuple_trap PASSED                               [100%]

=============================== warnings summary ===============================
test_nb01_tryit.py:5
  /Users/ajinkyak/Desktop/JULY-WORK/a - training_assignments/explorable_codelesson/test_nb01_tryit.py:5: PytestAss

### What you learned
- Files `test_*.py` and functions `test_*` are auto-discovered — no boilerplate.
- Plain `assert` gives rich failure introspection for free.
- `!pytest -v` shows every test; `!pytest -k "expr"` filters by name.

Next up: **fixtures** — reusable setup and teardown via dependency injection.